# GP09 — Spatial Join

Πρόβλεψε → Εκτέλεσε → Έλεγξε → Ερμήνευσε. Συνδέεις υπηρεσίες με διοικητικούς τομείς χωρίς να κόψεις τα σημεία.


## 1. Προετοιμασία

Το πρώτο κελί κατεβάζει μόνο τα απαιτούμενα δεδομένα όταν δεν υπάρχει έγκυρη cache. Εκτέλεσέ το πρώτο.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv, json
candidates=(Path('downloads/python/data'),Path('data'),Path('../../data'))
DATA_ROOT=next((p for p in candidates if (p/'gp09').is_dir()),Path('data'))
DATA_ROOT.mkdir(parents=True,exist_ok=True)
DATA_URLS={'services.csv':'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp09/services.csv','districts.geojson':'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp09/districts.geojson'}
REQUIRED={'services.csv':{'service_id','name','service_type','longitude','latitude'},'districts.geojson':{'district_id','district_name'}}
def valid_cached(path,required):
    if not path.exists() or path.stat().st_size==0:return False
    try:
        if path.suffix=='.csv':
            with path.open(encoding='utf-8-sig',newline='') as f:
                r=csv.DictReader(f);return required.issubset(r.fieldnames or []) and next(r,None) is not None
        d=json.loads(path.read_text(encoding='utf-8'));return d.get('type')=='FeatureCollection' and all('geometry' in f and required.issubset(f.get('properties',{})) for f in d.get('features',[]))
    except Exception:return False
for filename,url in DATA_URLS.items():
    target=DATA_ROOT/'gp09'/filename
    if valid_cached(target,REQUIRED[filename]):print(f'[OK] Ήδη διαθέσιμο: {target}');continue
    target.parent.mkdir(parents=True,exist_ok=True)
    try:
        with urlopen(url,timeout=30) as r:target.write_bytes(r.read())
        if not valid_cached(target,REQUIRED[filename]):raise RuntimeError('μη έγκυρη δομή ή πεδία')
        print(f'[OK] Κατέβηκε: {target}')
    except Exception as exc:
        if target.exists():target.unlink()
        raise RuntimeError(f'Αποτυχία προετοιμασίας: {url} — {exc}') from exc


## 2. Inputs και πρόβλεψη

Πρόβλεψε geometry, left/right, CRS και πλήθη πριν τον κώδικα.


In [ ]:
import pandas as pd
import geopandas as gpd
root=DATA_ROOT/'gp09'
df=pd.read_csv(root/'services.csv')
services=gpd.GeoDataFrame(df,geometry=gpd.points_from_xy(df.longitude,df.latitude),crs='EPSG:4326')
districts=gpd.read_file(root/'districts.geojson')
assert len(services)==6 and len(districts)==3
assert services.geometry.geom_type.eq('Point').all() and districts.geometry.geom_type.eq('Polygon').all()
assert services.crs.to_epsg()==districts.crs.to_epsg()==4326
print(services.columns.tolist(),districts.columns.tolist(),services.total_bounds,districts.total_bounds)


## 3. Guided left spatial join

Το `services` είναι left, το `districts` right και το `within` ελέγχει point μέσα σε polygon.


In [ ]:
joined_left=gpd.sjoin(services,districts,how='left',predicate='within')
assert len(joined_left)==6 and joined_left.geometry.geom_type.eq('Point').all() and joined_left.crs.to_epsg()==4326
assert joined_left['district_id'].notna().sum()==5
print(joined_left[['service_id','district_id']])


## 4. Matched, unmatched και map

Η S06 είναι η εσκεμμένα unmatched υπηρεσία. Αυτό μπορεί να είναι έγκυρη πληροφορία και όχι κατεστραμμένο δεδομένο.


In [ ]:
unmatched=joined_left[joined_left['district_id'].isna()]
assert unmatched['service_id'].tolist()==['S06']
ax=districts.boundary.plot(figsize=(7,5),color='0.4')
services.plot(ax=ax,color='black',markersize=25)
unmatched.plot(ax=ax,color='red',markersize=45)


## 5. Controlled left/inner comparison

Σύγκρινε rows, unique IDs και unmatched εγγραφές. Η άσκηση απαιτεί αιτιολόγηση, αλλά δεν πρέπει να εμποδίζει το guided workflow.


In [ ]:
# TODO: συμπλήρωσε πραγματικές συγκρίσεις και αιτιολόγηση.
joined_inner=gpd.sjoin(services,districts,how='inner',predicate='within')
rows_comparison=None
unique_id_check=None
unmatched_check=None
if rows_comparison is None or unique_id_check is None or unmatched_check is None:raise NotImplementedError('Σύγκρινε left/inner rows, unique IDs και unmatched')
assert rows_comparison and unique_id_check and unmatched_check


## 6. within, intersects και όρια

Στο προσωρινό οριακό point το `within` δίνει κανένα match, ενώ το `intersects` δίνει δύο. Το παράδειγμα δεν αλλάζει τα δεδομένα.


In [ ]:
from shapely.geometry import Point,Polygon
boundary_point=gpd.GeoDataFrame({'service_id':['TEMP']},geometry=[Point(23.58,41.10)],crs=services.crs)
within_demo=gpd.sjoin(boundary_point,districts,how='left',predicate='within')
intersects_demo=gpd.sjoin(boundary_point,districts,how='left',predicate='intersects')
assert within_demo['district_id'].notna().sum()==0 and intersects_demo['district_id'].notna().sum()==2


## 7. One-to-many ως χωριστό παράδειγμα

Τα πραγματικά districts δεν επικαλύπτονται. Δες την έννοια με δύο προσωρινά overlapping polygons.


In [ ]:
demo_points=gpd.GeoDataFrame({'id':['P1']},geometry=[Point(.5,.5)],crs='EPSG:4326')
demo_polys=gpd.GeoDataFrame({'zone':['A','B']},geometry=[Polygon([(0,0),(1,0),(1,1),(0,1)]),Polygon([(.25,.25),(1.25,.25),(1.25,1.25),(.25,1.25)])],crs='EPSG:4326')
demo_join=gpd.sjoin(demo_points,demo_polys,how='left',predicate='within')
assert len(demo_join)==2


## 8. Guided aggregation και export

Το guided summary μετρά υπηρεσίες ανά διοικητικό τομέα. Η λεπτομερής αναφορά ανά κατηγορία είναι διαφορετική εργασία του Mini Challenge.


In [ ]:
# Ανεξάρτητο guided κελί: δεν εξαρτάται από την ελεγχόμενη άσκηση gp09-11.
joined_inner=gpd.sjoin(services,districts,how='inner',predicate='within')
counts=joined_inner.groupby(['district_id','district_name']).size().reset_index(name='service_count')
assert len(joined_inner)==5 and counts['service_count'].sum()==5
joined_left.to_file('gp09_services_by_district.gpkg',layer='services',driver='GPKG')
reloaded=gpd.read_file('gp09_services_by_district.gpkg',layer='services')
assert len(reloaded)==len(joined_left) and reloaded.crs.to_epsg()==4326
assert reloaded.geometry.geom_type.eq('Point').all() and reloaded.geometry.notna().all() and reloaded.geometry.is_valid.all() and (~reloaded.geometry.is_empty).all()
assert set(reloaded['service_id'])==set(joined_left['service_id']) and (reloaded.total_bounds-joined_left.total_bounds).__pow__(2).max()<1e-8
counts.to_csv('gp09_service_counts.csv',index=False)


## 9. Ανεξάρτητο Mini Challenge

Δημιούργησε report ανά district και service type και ξεχωριστό unmatched report. Διάλεξε μόνος σου οργάνωση, aggregation, QA και ερμηνεία.

Το επόμενο κελί είναι σκόπιμα ημιτελές και δεν αποτελεί έτοιμη λύση.


In [ ]:
raise NotImplementedError('Συμπλήρωσε το ανεξάρτητο Mini Challenge')


## 10. Readiness checkpoint

Μπορείς να εξηγήσεις τι λύνεις, γιατί επιλέγεις predicate/join type, τι έδειξαν counts και nulls, και τι δεν αποδεικνύει ένα assertion;
